# Comparação de F1-Score entre Múltiplas Versões

Este notebook permite comparar a evolução do F1-Score (por pixel e por componente) entre diferentes versões do experimento ao longo das iterações.

**Funcionalidades:**
- Comparação de múltiplas versões simultaneamente
- Visualização da evolução do F1-Score por pixel e por componente
- Análise comparativa de desempenho entre versões
- Estatísticas resumidas por versão

## Setup e Imports

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100

# Paleta de cores para versões (expandível)
VERSION_COLORS = [
    '#2E86AB',  # Azul
    '#D62828',  # Vermelho
    '#06A77D',  # Verde
    '#F18F01',  # Laranja
    '#A23B72',  # Roxo
    '#F77F00',  # Laranja escuro
    '#6A4C93',  # Roxo escuro
    '#118AB2',  # Azul claro
]

print("✓ Imports realizados com sucesso!")

## Configuração de Versões

Defina aqui as versões que deseja comparar. Cada entrada deve ser uma tupla com:
- Nome da versão (para exibição)
- Caminho relativo ou absoluto para a pasta da versão

In [ ]:
# Defina aqui as versões para comparação
# Formato: (nome_exibição, caminho_para_pasta)
VERSIONS = [
    # ("v01", "../../../bioflore_data/v01"),
    ("v02", "../../../bioflore_data/v02"),
    ("v03", "../../../bioflore_data/v03"),
    ("v04", "../../../bioflore_data/v04"),
]

# Converter caminhos para Path objects e validar
version_paths = {}
version_args = {}

for version_name, version_path in VERSIONS:
    path_obj = Path(version_path)
    if not path_obj.exists():
        print(f"⚠️  Aviso: Caminho não encontrado para {version_name}: {path_obj}")
        continue
    
    # Carregar args.yaml se existir
    args_path = path_obj / "args.yaml"
    if args_path.exists():
        with open(args_path, 'r') as f:
            version_args[version_name] = yaml.safe_load(f)
        version_paths[version_name] = path_obj
        num_iter = version_args[version_name].get('num_iter', 0)
        print(f"✓ {version_name}: {path_obj} (iterações: {num_iter})")
    else:
        print(f"⚠️  Aviso: args.yaml não encontrado para {version_name}")

print(f"\n✓ Total de versões carregadas: {len(version_paths)}")

## Funções de Carregamento

In [ ]:
def load_global_metrics(data_path, num_iter, version_name):
    """
    Carrega métricas globais de todas as iterações para uma versão.
    
    Args:
        data_path: Path para a pasta da versão
        num_iter: Número máximo de iterações
        version_name: Nome da versão (para adicionar como coluna)
    
    Returns:
        DataFrame com métricas globais por iteração, incluindo coluna 'version'
    """
    metrics_list = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        metrics_file = iter_folder / "global_metrics.yaml"
        
        if not metrics_file.exists():
            # Pular iterações sem métricas (ex: iter_000 pode não ter)
            continue
        
        try:
            with open(metrics_file, 'r') as f:
                metrics = yaml.safe_load(f)
            
            # Extrair métricas de treino e teste
            iter_data = {'iter': i, 'version': version_name}
            
            # Métricas de treino
            if 'global/train/Accuracy' in metrics:
                iter_data['train_accuracy'] = metrics['global/train/Accuracy']
                iter_data['train_kappa'] = metrics['global/train/KappaScore']
                iter_data['train_f1_avg'] = metrics['global/train/avgF1']
                iter_data['train_f1_weighted'] = metrics['global/train/avgF1_weighted']
                
                # F1 por classe
                if 'global/train/F1' in metrics:
                    for idx, f1 in enumerate(metrics['global/train/F1'], 1):
                        iter_data[f'train_f1_class_{idx}'] = f1
            
            # Métricas de teste
            if 'global/test/Accuracy' in metrics:
                iter_data['test_accuracy'] = metrics['global/test/Accuracy']
                iter_data['test_kappa'] = metrics['global/test/KappaScore']
                iter_data['test_f1_avg'] = metrics['global/test/avgF1']
                iter_data['test_f1_weighted'] = metrics['global/test/avgF1_weighted']
                
                # F1 por classe
                if 'global/test/F1' in metrics:
                    for idx, f1 in enumerate(metrics['global/test/F1'], 1):
                        iter_data[f'test_f1_class_{idx}'] = f1
            
            metrics_list.append(iter_data)
        except Exception as e:
            print(f"⚠️  Erro ao carregar {metrics_file}: {e}")
            continue
    
    if not metrics_list:
        return pd.DataFrame()
    
    df = pd.DataFrame(metrics_list)
    return df


def load_component_metrics(data_path, num_iter, version_name):
    """
    Carrega métricas de componentes de todas as iterações para uma versão.
    
    Args:
        data_path: Path para a pasta da versão
        num_iter: Número máximo de iterações
        version_name: Nome da versão (para adicionar como coluna)
    
    Returns:
        DataFrame com métricas de componentes por iteração, incluindo coluna 'version'
    """
    metrics_list = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        metrics_file = iter_folder / "global_component_metrics.yaml"
        
        if not metrics_file.exists():
            # Pular iterações sem métricas de componente
            continue
        
        try:
            with open(metrics_file, 'r') as f:
                metrics = yaml.safe_load(f)
            
            # Extrair métricas de componente
            iter_data = {
                'iter': i,
                'version': version_name,
                'component_accuracy': metrics.get('global/component/Accuracy', 0),
                'component_f1_avg': metrics.get('global/component/avgF1', 0),
                'component_avg_prec': metrics.get('global/component/avgPrec', 0),
                'component_avg_rec': metrics.get('global/component/avgRec', 0),
                'n_components': metrics.get('global/component/n_components', 0)
            }
            
            # F1 por classe (componente)
            if 'global/component/F1' in metrics:
                for idx, f1 in enumerate(metrics['global/component/F1'], 1):
                    iter_data[f'component_f1_class_{idx}'] = f1
            
            metrics_list.append(iter_data)
        except Exception as e:
            print(f"⚠️  Erro ao carregar {metrics_file}: {e}")
            continue
    
    if not metrics_list:
        return pd.DataFrame()
    
    df = pd.DataFrame(metrics_list)
    return df

print("✓ Funções de carregamento definidas")

## Carregamento de Dados de Todas as Versões

In [ ]:
# Carregar métricas de todas as versões
all_global_metrics = []
all_component_metrics = []

for version_name, data_path in version_paths.items():
    num_iter = version_args[version_name].get('num_iter', 0)
    
    print(f"\n📊 Carregando {version_name}...")
    
    # Carregar métricas globais
    df_global = load_global_metrics(data_path, num_iter, version_name)
    if not df_global.empty:
        all_global_metrics.append(df_global)
        print(f"   ✓ Métricas globais: {len(df_global)} iterações")
    else:
        print(f"   ⚠️  Nenhuma métrica global encontrada")
    
    # Carregar métricas de componente
    df_component = load_component_metrics(data_path, num_iter, version_name)
    if not df_component.empty:
        all_component_metrics.append(df_component)
        print(f"   ✓ Métricas de componente: {len(df_component)} iterações")
    else:
        print(f"   ⚠️  Nenhuma métrica de componente encontrada")

# Combinar todos os DataFrames
if all_global_metrics:
    df_all_global = pd.concat(all_global_metrics, ignore_index=True)
    print(f"\n✓ Total de métricas globais carregadas: {len(df_all_global)} registros")
    print(f"  Versões: {df_all_global['version'].unique().tolist()}")
else:
    df_all_global = pd.DataFrame()
    print("\n⚠️  Nenhuma métrica global foi carregada")

if all_component_metrics:
    df_all_component = pd.concat(all_component_metrics, ignore_index=True)
    print(f"✓ Total de métricas de componente carregadas: {len(df_all_component)} registros")
    print(f"  Versões: {df_all_component['version'].unique().tolist()}")
else:
    df_all_component = pd.DataFrame()
    print("⚠️  Nenhuma métrica de componente foi carregada")

## Visualizações Comparativas

### Gráfico 1: F1-Score por Pixel (Teste) - Comparação entre Versões

In [ ]:
if not df_all_global.empty and 'test_f1_avg' in df_all_global.columns:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    versions = df_all_global['version'].unique()
    
    for idx, version in enumerate(versions):
        df_version = df_all_global[df_all_global['version'] == version].sort_values('iter')
        color = VERSION_COLORS[idx % len(VERSION_COLORS)]
        
        ax.plot(df_version['iter'], df_version['test_f1_avg'], 
                marker='o', linewidth=2.5, markersize=8, 
                label=version, color=color)
    
    ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
    ax.set_title('Comparação: F1-Score por Pixel (Teste) entre Versões', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # Ajustar limites do eixo X
    if not df_all_global.empty:
        max_iter = df_all_global['iter'].max()
        ax.set_xlim(-0.5, max_iter + 0.5)
    
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Dados insuficientes para gerar gráfico de F1-Score por pixel")

### Gráfico 2: F1-Score por Componente - Comparação entre Versões

In [ ]:
if not df_all_component.empty and 'component_f1_avg' in df_all_component.columns:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    versions = df_all_component['version'].unique()
    
    for idx, version in enumerate(versions):
        df_version = df_all_component[df_all_component['version'] == version].sort_values('iter')
        color = VERSION_COLORS[idx % len(VERSION_COLORS)]
        
        ax.plot(df_version['iter'], df_version['component_f1_avg'], 
                marker='D', linewidth=2.5, markersize=8, 
                label=version, color=color)
    
    ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
    ax.set_title('Comparação: F1-Score por Componente entre Versões', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # Ajustar limites do eixo X
    if not df_all_component.empty:
        max_iter = df_all_component['iter'].max()
        ax.set_xlim(-0.5, max_iter + 0.5)
    
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Dados insuficientes para gerar gráfico de F1-Score por componente")

### Gráfico 3: Comparação Lado a Lado - Pixel vs Componente por Versão

In [ ]:
if not df_all_global.empty and not df_all_component.empty:
    versions = sorted(set(df_all_global['version'].unique()) & set(df_all_component['version'].unique()))
    
    if versions:
        n_versions = len(versions)
        fig, axes = plt.subplots(1, n_versions, figsize=(6 * n_versions, 6))
        
        if n_versions == 1:
            axes = [axes]
        
        for idx, version in enumerate(versions):
            ax = axes[idx]
            color = VERSION_COLORS[idx % len(VERSION_COLORS)]
            
            # F1-Score por pixel
            df_global_version = df_all_global[df_all_global['version'] == version].sort_values('iter')
            if not df_global_version.empty and 'test_f1_avg' in df_global_version.columns:
                ax.plot(df_global_version['iter'], df_global_version['test_f1_avg'], 
                        marker='o', linewidth=2.5, markersize=8, 
                        label='F1-Score (Pixel)', color=color, linestyle='-')
            
            # F1-Score por componente
            df_component_version = df_all_component[df_all_component['version'] == version].sort_values('iter')
            if not df_component_version.empty and 'component_f1_avg' in df_component_version.columns:
                ax.plot(df_component_version['iter'], df_component_version['component_f1_avg'], 
                        marker='D', linewidth=2.5, markersize=8, 
                        label='F1-Score (Componente)', color=color, linestyle='--', alpha=0.7)
            
            ax.set_xlabel('Iteração', fontsize=11, fontweight='bold')
            ax.set_ylabel('F1-Score Médio (%)', fontsize=11, fontweight='bold')
            ax.set_title(f'{version}\nPixel vs Componente', fontsize=12, fontweight='bold')
            ax.legend(loc='best', fontsize=10, framealpha=0.9)
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 100)
            
            if not df_global_version.empty:
                max_iter = max(df_global_version['iter'].max(), 
                              df_component_version['iter'].max() if not df_component_version.empty else 0)
                ax.set_xlim(-0.5, max_iter + 0.5)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Nenhuma versão com dados de pixel e componente disponíveis simultaneamente")
else:
    print("⚠️  Dados insuficientes para gerar comparação lado a lado")

### Gráfico 4: Comparação Unificada - Todas as Versões (Pixel e Componente)

In [ ]:
def load_region_metrics(data_path, num_iter, version_name, num_regions=3):
    """
    Carrega métricas por pixel para cada região de todas as iterações para uma versão.
    
    Args:
        data_path: Path para a pasta da versão
        num_iter: Número máximo de iterações
        version_name: Nome da versão (para adicionar como coluna)
        num_regions: Número de regiões (padrão: 3)
    
    Returns:
        DataFrame com métricas por pixel por região, incluindo colunas: iter, version, region, test_f1_avg
    """
    metrics_list = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        
        if not iter_folder.exists():
            continue
        
        # Detectar número de regiões verificando existência de pastas
        detected_regions = []
        for r in range(num_regions):
            region_folder = iter_folder / f"region_{r}"
            if region_folder.exists():
                detected_regions.append(r)
        
        # Se não encontrou nenhuma região, pular iteração
        if not detected_regions:
            continue
        
        for region_idx in detected_regions:
            region_folder = iter_folder / f"region_{region_idx}"
            metrics_file = region_folder / "test_metrics.yaml"
            
            if not metrics_file.exists():
                # Pular regiões sem métricas
                continue
            
            try:
                with open(metrics_file, 'r') as f:
                    metrics = yaml.safe_load(f)
                
                # Extrair F1 por pixel
                f1_key = f"region_{region_idx}/test/avgF1"
                f1_value = metrics.get(f1_key, 0)
                
                iter_data = {
                    'iter': i,
                    'version': version_name,
                    'region': region_idx,
                    'test_f1_avg': f1_value
                }
                
                metrics_list.append(iter_data)
            except Exception as e:
                print(f"⚠️  Erro ao carregar {metrics_file}: {e}")
                continue
    
    if not metrics_list:
        return pd.DataFrame()
    
    df = pd.DataFrame(metrics_list)
    return df

In [ ]:
# Carregar métricas por região para todas as versões
all_region_metrics = []

for version_name, data_path in version_paths.items():
    num_iter = version_args[version_name].get('num_iter', 0)
    
    print(f"\n📊 Carregando métricas por região para {version_name}...")
    
    # Detectar número de regiões verificando primeira iteração
    num_regions = 3  # padrão
    first_iter = data_path / "iter_000"
    if first_iter.exists():
        detected_regions = []
        for r in range(10):  # verificar até 10 regiões
            region_folder = first_iter / f"region_{r}"
            if region_folder.exists():
                detected_regions.append(r)
        if detected_regions:
            num_regions = max(detected_regions) + 1
    
    # Carregar métricas por região
    df_region = load_region_metrics(data_path, num_iter, version_name, num_regions)
    if not df_region.empty:
        all_region_metrics.append(df_region)
        regions_found = sorted(df_region['region'].unique())
        print(f"   ✓ Métricas por região: {len(df_region)} registros (regiões: {regions_found})")
    else:
        print(f"   ⚠️  Nenhuma métrica por região encontrada")

# Combinar todos os DataFrames
if all_region_metrics:
    df_all_regions = pd.concat(all_region_metrics, ignore_index=True)
    print(f"\n✓ Total de métricas por região carregadas: {len(df_all_regions)} registros")
    print(f"  Versões: {df_all_regions['version'].unique().tolist()}")
    print(f"  Regiões: {sorted(df_all_regions['region'].unique())}")
else:
    df_all_regions = pd.DataFrame()
    print("\n⚠️  Nenhuma métrica por região foi carregada")

In [ ]:
if not df_all_global.empty and not df_all_component.empty:
    fig, ax = plt.subplots(figsize=(16, 8))
    
    versions = sorted(set(df_all_global['version'].unique()) & set(df_all_component['version'].unique()))
    
    for idx, version in enumerate(versions):
        color = VERSION_COLORS[idx % len(VERSION_COLORS)]
        
        # F1-Score por pixel
        df_global_version = df_all_global[df_all_global['version'] == version].sort_values('iter')
        if not df_global_version.empty and 'test_f1_avg' in df_global_version.columns:
            ax.plot(df_global_version['iter'], df_global_version['test_f1_avg'], 
                    marker='o', linewidth=2.5, markersize=8, 
                    label=f'{version} (Pixel)', color=color, linestyle='-')
        
        # F1-Score por componente
        df_component_version = df_all_component[df_all_component['version'] == version].sort_values('iter')
        if not df_component_version.empty and 'component_f1_avg' in df_component_version.columns:
            ax.plot(df_component_version['iter'], df_component_version['component_f1_avg'], 
                    marker='D', linewidth=2.5, markersize=8, 
                    label=f'{version} (Componente)', color=color, linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
    ax.set_title('Comparação Completa: F1-Score por Pixel e Componente - Todas as Versões', 
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10, framealpha=0.9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    
    if not df_all_global.empty:
        max_iter = max(df_all_global['iter'].max(), 
                      df_all_component['iter'].max() if not df_all_component.empty else 0)
        ax.set_xlim(-0.5, max_iter + 0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Dados insuficientes para gerar comparação unificada")

## Comparação por Região

### Gráfico 1: F1-Score por Pixel por Região

In [ ]:
if not df_all_regions.empty and 'test_f1_avg' in df_all_regions.columns:
    # Detectar número de regiões disponíveis
    regions = sorted(df_all_regions['region'].unique())
    num_regions = len(regions)
    
    if num_regions > 0:
        fig, axes = plt.subplots(1, num_regions, figsize=(5*num_regions, 6))
        
        # Se houver apenas uma região, axes não será um array
        if num_regions == 1:
            axes = [axes]
        
        versions = df_all_regions['version'].unique()
        
        for region_idx, ax in zip(regions, axes):
            df_region = df_all_regions[df_all_regions['region'] == region_idx]
            
            for idx, version in enumerate(versions):
                df_version = df_region[df_region['version'] == version].sort_values('iter')
                if not df_version.empty:
                    color = VERSION_COLORS[idx % len(VERSION_COLORS)]
                    ax.plot(df_version['iter'], df_version['test_f1_avg'], 
                            marker='o', linewidth=2.5, markersize=8, 
                            label=version, color=color)
            
            ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
            ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
            ax.set_title(f'Região {region_idx}', fontsize=13, fontweight='bold')
            ax.legend(loc='best', fontsize=10, framealpha=0.9)
            ax.grid(True, alpha=0.3)
            
            # Ajustar limites do eixo X
            if not df_region.empty:
                max_iter = df_region['iter'].max()
                ax.set_xlim(-0.5, max_iter + 0.5)
            
            ax.set_ylim(0, 100)
        
        fig.suptitle('F1-Score por Pixel (Teste) - Comparação por Região', 
                     fontsize=15, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Nenhuma região encontrada nos dados")
else:
    print("⚠️  Dados insuficientes para gerar gráfico de F1-Score por região")

In [ ]:
# Parâmetro para selecionar quais versões plotar
# selected_versions deve ser uma lista de strings, ex: ['v02', 'v04']
# Se selected_versions não estiver definido, plota todas
selected_versions = ["v04"]
if not df_all_regions.empty and 'test_f1_avg' in df_all_regions.columns:
    # Detectar número de regiões disponíveis
    regions = sorted(df_all_regions['region'].unique())
    num_regions = len(regions)
    
    if num_regions > 0:
        fig, axes = plt.subplots(1, num_regions, figsize=(5*num_regions, 6))
        
        # Se houver apenas uma região, axes não será um array
        if num_regions == 1:
            axes = [axes]
        
        all_versions = df_all_regions['version'].unique()
        # Filtrar versões selecionadas se o parâmetro for fornecido
        if selected_versions is not None:
            versions = [v for v in selected_versions if v in all_versions]
        else:
            versions = all_versions
        
        for region_idx, ax in zip(regions, axes):
            df_region = df_all_regions[df_all_regions['region'] == region_idx]
            
            for idx, version in enumerate(versions):
                df_version = df_region[df_region['version'] == version].sort_values('iter')
                if not df_version.empty:
                    color = VERSION_COLORS[idx % len(VERSION_COLORS)]
                    ax.plot(df_version['iter'], df_version['test_f1_avg'], 
                            marker='o', linewidth=2.5, markersize=8, 
                            label=version, color=color)
            
            ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
            ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
            ax.set_title(f'Região {region_idx}', fontsize=13, fontweight='bold')
            ax.legend(loc='best', fontsize=10, framealpha=0.9)
            ax.grid(True, alpha=0.3)
            
            # Ajustar limites do eixo X
            if not df_region.empty:
                max_iter = df_region['iter'].max()
                ax.set_xlim(-0.5, max_iter + 0.5)
            
            ax.set_ylim(0, 100)
        
        fig.suptitle('F1-Score por Pixel (Teste) - Comparação por Região', 
                     fontsize=15, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Nenhuma região encontrada nos dados")
else:
    print("⚠️  Dados insuficientes para gerar gráfico de F1-Score por região")

In [ ]:
selected_versions

In [ ]:
v for v in selected_versions if v in available_versions

### Gráfico 5: Diferença entre Pixel e Componente por Versão

In [ ]:
if not df_all_global.empty and not df_all_component.empty:
    # Mesclar dados para calcular diferença
    merged = pd.merge(
        df_all_global[['version', 'iter', 'test_f1_avg']], 
        df_all_component[['version', 'iter', 'component_f1_avg']], 
        on=['version', 'iter'], 
        how='inner'
    )
    
    if not merged.empty:
        merged['difference'] = merged['test_f1_avg'] - merged['component_f1_avg']
        
        fig, ax = plt.subplots(figsize=(14, 7))
        
        versions = merged['version'].unique()
        
        for idx, version in enumerate(versions):
            df_version = merged[merged['version'] == version].sort_values('iter')
            color = VERSION_COLORS[idx % len(VERSION_COLORS)]
            
            ax.plot(df_version['iter'], df_version['difference'], 
                    marker='s', linewidth=2.5, markersize=8, 
                    label=version, color=color)
        
        # Linha de referência em zero
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
        ax.set_ylabel('Diferença (Pixel - Componente)', fontsize=12, fontweight='bold')
        ax.set_title('Diferença entre F1-Score por Pixel e por Componente', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=11, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        
        max_iter = merged['iter'].max()
        ax.set_xlim(-0.5, max_iter + 0.5)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Não foi possível mesclar dados de pixel e componente")
else:
    print("⚠️  Dados insuficientes para calcular diferença")

## Estatísticas Comparativas

### Resumo por Versão (Última Iteração)

In [ ]:
print("\n" + "="*80)
print("📊 COMPARAÇÃO DE F1-SCORE - ÚLTIMA ITERAÇÃO")
print("="*80)

summary_data = []

for version_name in sorted(version_paths.keys()):
    summary = {'Versão': version_name}
    
    # Métricas globais (pixel)
    if not df_all_global.empty:
        df_version_global = df_all_global[df_all_global['version'] == version_name]
        if not df_version_global.empty:
            last_iter_global = df_version_global['iter'].max()
            last_data_global = df_version_global[df_version_global['iter'] == last_iter_global]
            
            if not last_data_global.empty and 'test_f1_avg' in last_data_global.columns:
                summary['Última Iteração'] = int(last_iter_global)
                summary['F1-Score Pixel (%)'] = f"{last_data_global['test_f1_avg'].values[0]:.2f}"
                summary['Acurácia Pixel (%)'] = f"{last_data_global['test_accuracy'].values[0]:.2f}" if 'test_accuracy' in last_data_global.columns else "N/A"
    
    # Métricas de componente
    if not df_all_component.empty:
        df_version_component = df_all_component[df_all_component['version'] == version_name]
        if not df_version_component.empty:
            last_iter_component = df_version_component['iter'].max()
            last_data_component = df_version_component[df_version_component['iter'] == last_iter_component]
            
            if not last_data_component.empty and 'component_f1_avg' in last_data_component.columns:
                summary['F1-Score Componente (%)'] = f"{last_data_component['component_f1_avg'].values[0]:.2f}"
                summary['Acurácia Componente (%)'] = f"{last_data_component['component_accuracy'].values[0]:.2f}" if 'component_accuracy' in last_data_component.columns else "N/A"
    
    # Calcular diferença se ambos disponíveis
    if 'F1-Score Pixel (%)' in summary and 'F1-Score Componente (%)' in summary:
        pixel_f1 = float(summary['F1-Score Pixel (%)'])
        component_f1 = float(summary['F1-Score Componente (%)'])
        summary['Diferença (Pixel - Componente)'] = f"{pixel_f1 - component_f1:.2f}"
    
    summary_data.append(summary)

# Criar DataFrame de resumo
if summary_data:
    df_summary = pd.DataFrame(summary_data)
    print("\n")
    print(df_summary.to_string(index=False))
    print("\n")
else:
    print("⚠️  Nenhum dado disponível para resumo")

### Melhor Versão por Métrica

In [ ]:
print("\n" + "="*80)
print("🏆 MELHOR VERSÃO POR MÉTRICA")
print("="*80)

best_versions = {}

# Melhor F1-Score por Pixel
if not df_all_global.empty and 'test_f1_avg' in df_all_global.columns:
    # Pegar última iteração de cada versão
    last_iters_global = df_all_global.groupby('version')['iter'].max()
    best_pixel_f1 = -1
    best_pixel_version = None
    
    for version_name in df_all_global['version'].unique():
        last_iter = last_iters_global[version_name]
        last_data = df_all_global[(df_all_global['version'] == version_name) & 
                                  (df_all_global['iter'] == last_iter)]
        if not last_data.empty:
            f1_value = last_data['test_f1_avg'].values[0]
            if f1_value > best_pixel_f1:
                best_pixel_f1 = f1_value
                best_pixel_version = version_name
    
    if best_pixel_version:
        best_versions['F1-Score Pixel'] = (best_pixel_version, best_pixel_f1)
        print(f"\n✓ Melhor F1-Score por Pixel: {best_pixel_version} ({best_pixel_f1:.2f}%)")

# Melhor F1-Score por Componente
if not df_all_component.empty and 'component_f1_avg' in df_all_component.columns:
    last_iters_component = df_all_component.groupby('version')['iter'].max()
    best_component_f1 = -1
    best_component_version = None
    
    for version_name in df_all_component['version'].unique():
        last_iter = last_iters_component[version_name]
        last_data = df_all_component[(df_all_component['version'] == version_name) & 
                                    (df_all_component['iter'] == last_iter)]
        if not last_data.empty:
            f1_value = last_data['component_f1_avg'].values[0]
            if f1_value > best_component_f1:
                best_component_f1 = f1_value
                best_component_version = version_name
    
    if best_component_version:
        best_versions['F1-Score Componente'] = (best_component_version, best_component_f1)
        print(f"✓ Melhor F1-Score por Componente: {best_component_version} ({best_component_f1:.2f}%)")

print("\n")

### Tabela Detalhada: Evolução por Versão

In [ ]:
# Criar tabela comparativa detalhada
if not df_all_global.empty:
    print("\n📋 EVOLUÇÃO DO F1-SCORE POR PIXEL (TESTE)")
    print("="*80)
    
    for version_name in sorted(df_all_global['version'].unique()):
        df_version = df_all_global[df_all_global['version'] == version_name].sort_values('iter')
        print(f"\n{version_name}:")
        print(df_version[['iter', 'test_f1_avg', 'test_accuracy', 'test_kappa']].to_string(index=False))

if not df_all_component.empty:
    print("\n\n📋 EVOLUÇÃO DO F1-SCORE POR COMPONENTE")
    print("="*80)
    
    for version_name in sorted(df_all_component['version'].unique()):
        df_version = df_all_component[df_all_component['version'] == version_name].sort_values('iter')
        print(f"\n{version_name}:")
        print(df_version[['iter', 'component_f1_avg', 'component_accuracy', 'n_components']].to_string(index=False))